# 🔬 Monitor RP Bench-Test Notebook

**Purpose:** Verify that the **Mon RedPitaya** (`192.168.0.99`) works correctly in isolation.  
The Cav and Lock1 boards are **not required**. A function generator feeds IN1/IN2 directly.

**What this notebook tests:**
- SSH upload of RP-side scripts and connection handshake
- `start_monitor()` — live cavity-signal plot (raw IN1/IN2 trace)
- `start_error_monitor()` — live error-vs-time plot
- Clean shutdown

> **Wiring:** Connect your function generator output(s) to the Mon RP SMA inputs (IN1 and/or IN2).  
> Any signal works — a sine or triangle wave is fine for a visual check.

---

---
## Phase 0: Configuration

Edit only the cell below. All other cells use these variables automatically.

In [ ]:
# ── Board address ──────────────────────────────────────────────────────────────
RP_MON_IP = "192.168.0.99"   # Mon — monitor RP
RP_CAV_IP = "192.168.0.201"  # Cav — Scan RP

SSH_USER  = "root"
SSH_PASS  = "root"

print("Configuration loaded.")
print("  Mon  :", RP_MON_IP)

Configuration loaded.
  Mon  : 192.168.0.99


---
## Phase 1: Upload & Connect

1. Adds the repo root to `sys.path`.
2. Instantiates `LockClient` — SSHes into Mon, uploads RP-side scripts, loads settings.
3. Starts `RunLock.py` on Mon via SSH; waits for the FPGA overlay to initialise.
4. Starts the PC-side selector event loop in a background thread.

> **If this cell hangs beyond ~45 s:** SSH in manually and run:  
> `PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH python3 /root/RunLock.py`  
> to see the error. Common causes: port 5000 already in use (`pkill -f RunLock.py`),  
> FPGA overlay not loaded, or `rp` module not importable.

In [2]:
import sys, pathlib, threading, time

# ── Locate repo root and add to path ──────────────────────────────────────────
_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

from lockclient import LockClient, RP_client

Repo root: C:\Users\RikteemBhowmick\Projects 2025\RP-STCL


In [3]:
# ── Build RP_client dictionary (Mon only) ─────────────────────────────────────
RPs = {
    "Mon": RP_client((RP_MON_IP, 5000), {}, mode="monitor"),
}

print("Uploading scripts and loading settings...")
Lock = LockClient(RPs)
print("Done.")

Uploading scripts and loading settings...
Done.


In [4]:
# ── Connect (starts RunLock.py on Mon via SSH) ────────────────────────────────
def _wrap(fn, err):
    try:
        fn()
    except Exception as exc:
        err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start()
    t.join(timeout=timeout_s)
    if t.is_alive():
        raise TimeoutError(
            "{} timed out after {}s.\n"
            "Troubleshooting:\n"
            "  1. SSH into board and run: PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH python3 /root/RunLock.py\n"
            "  2. Check port 5000 is free: ss -tlnp | grep 5000\n"
            "  3. Kill stale processes:    pkill -f RunLock.py\n"
            "  4. Power-cycle the board if nothing else works.".format(name, timeout_s)
        )
    if "exc" in err:
        raise RuntimeError("{} failed: {}".format(name, err["exc"]))

run_with_timeout(Lock.connect_all, timeout_s=45, name="connect_all")
print("All boards connected.")

connecting...
All boards connected.


In [5]:
# ── Start PC-side event loop ───────────────────────────────────────────────────
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    time.sleep(2)
    print("Event loop started.")
else:
    print("Event loop already running.")

for name, rp in Lock.RPs.items():
    status = "connected" if rp.connected else "DISCONNECTED"
    print("  {:6s}  {}  {}".format(name, rp.addr[0], status))

Event loop started.
  Mon     192.168.0.99  connected


---
## Phase 2: Cavity Signal Monitor

Opens a live matplotlib window showing the raw IN1/IN2 trace acquired by Mon.  
With a function generator on IN1 you should see your waveform updating in real time.

Run the **stop** cell before moving on.

In [6]:
# ── Bypass Cav dependency for standalone Mon test ─────────────────────────────
# We inject a fake "Cav" entry into Lock.RPs with mode="ext_scan".
# This satisfies ALL three checks in lockclient.py without touching any file:
#   1. check_cavity_scanned: self.RPs["Cav"].mode == "ext_scan"  ✓
#   2. find_slave_RPs: val.settings["Master"] lookup on fake Cav  ✓
#   3. retrieve_settings: reads fake Cav's settings["Master"] dict  ✓
#
# Mon's settings["Master"] stays as "Cav" — exactly as loaded from Mon.json.
# The fake Cav has no SSH address and will never be connected to.
# Remove this cell when the real Cav RP is back — nothing else changes.

import copy

# Pull the Master sub-dict from Mon's own settings (dec, range, lockpoint etc.)
# so retrieve_settings() can do its ms→index conversions correctly.
mon_master_block = copy.deepcopy(Lock.RPs["Mon"].settings["Master"])

fake_cav = RP_client(("0.0.0.0", 5000), {}, mode="ext_scan")
fake_cav.label = "Cav"
fake_cav.settings = {
    "Master": mon_master_block   # gives retrieve_settings() the dec it needs
}
fake_cav.connected = False
fake_cav.loop_running = False

Lock.RPs["Cav"] = fake_cav
print("Fake 'Cav' RP injected with mode='ext_scan' — all Cav checks bypassed.")
print("Mon.settings['Master'] =", Lock.RPs["Mon"].settings["Master"])

Fake 'Cav' RP injected with mode='ext_scan' — all Cav checks bypassed.
Mon.settings['Master'] = Cav


In [7]:
# ── Start cavity signal monitor ───────────────────────────────────────────────
Lock.RPs["Mon"].settings["Master"] = "ext_scan"
print("Mon master overridden to 'ext_scan' — Cav check bypassed.")
Lock.start_monitor("Mon")
print("Cavity monitor started. You should see a live plot window.")
print("Stop with the next cell when done.")

Mon master overridden to 'ext_scan' — Cav check bypassed.


KeyError: 'ext_scan'

In [ ]:
# ── Stop cavity signal monitor ────────────────────────────────────────────────
Lock.stop_monitor("Mon")
time.sleep(0.5)
print("Cavity monitor stopped.")

---
## Phase 3: Error Monitor

Opens a live plot showing frequency error vs. time (scaled by FSR, default 906 MHz).  
Without a real lock running, errors will be noisy — that is expected for a bench test.  
The goal here is to confirm the monitor starts, plots, and can be stopped cleanly.

> `tmin=20e-3` sets the minimum interval between samples to 20 ms.

In [ ]:
# ── Start error monitor ───────────────────────────────────────────────────────
Lock.stop_monitor("Mon")   # ensure cavity monitor is off first
time.sleep(0.5)
Lock.start_error_monitor("Mon", tmin=20e-3)
print("Error monitor started. Close the plot window or run the next cell to stop.")

In [ ]:
# ── Stop error monitor ────────────────────────────────────────────────────────
Lock.stop_monitor("Mon")
time.sleep(0.5)
print("Error monitor stopped.")

---
## Phase 4: Safe Shutdown

Always run this at the end of the session to release port 5000 on the board.

In [ ]:
# ── Full clean shutdown ────────────────────────────────────────────────────────
Lock.close()
print("All boards disconnected. Session closed.")
print()
print("Verify port 5000 is free on Mon:")
print("  ssh root@{} 'ss -tlnp | grep 5000'".format(RP_MON_IP))